# 03 · ISRO INSAT-3DR aerosol optical depth vs NASA
Data: INSAT-3DR Imager AOD at 550 nm, product **`3RIMG_L2G_AOD`**, 0.1° grid, every 30 min (daytime), from **MOSDAC** (ISRO Space Applications Centre). Downloaded with ISRO's official `mdapi` tool (free MOSDAC registration).

Two quirks found while downloading:
1. `config.json` needs the key **`username/email`** (not `username`), or login fails.
2. The tool only returns the first 100 results per search, so we request one week at a time (~49 files).

Output: `data/insat_site_daily_aod_2024.csv` (raw `.h5` files are ~0.9 GB and not stored in the repo).

In [ ]:
!wget -q https://mosdac.gov.in/software/mdapi.zip && unzip -o -q mdapi.zip
import json, getpass, glob, os, subprocess, datetime as dt
import pandas as pd, numpy as np, h5py
from scipy import stats
REPO = "https://raw.githubusercontent.com/ishjeetsingh/india-solar-soiling/main/"
d = os.path.dirname(os.path.abspath(glob.glob("/content/**/mdapi.py", recursive=True)[0]))
cfg = {
  "user_credentials": {"username/email": input("MOSDAC username: "), "password": getpass.getpass("MOSDAC password: ")},
  "search_parameters": {"datasetId": "3RIMG_L2G_AOD", "startTime": "", "endTime": "",
                        "count": "100", "boundingBox": "68.0,6.0,98.0,37.0", "gId": ""},
  "download_settings": {"download_path": "/content/insat", "organize_by_date": False,
                        "skip_user_prompt": True, "generate_error_log": True,
                        "error_log_path": "/content/insat_logs"}}

## Download 2024, one week at a time (~20 min)
Skips weeks already downloaded; file names contain the date (e.g. `3RIMG_15MAR2024_0845_L2G_AOD_V02R00.h5`).

In [ ]:
have = lambda: {os.path.basename(f)[6:15] for f in glob.glob("/content/insat/**/*.h5", recursive=True)}
start, stop = dt.date(2024, 1, 1), dt.date(2024, 12, 31)
while start <= stop:
    end = min(start + dt.timedelta(days=6), stop)
    days = [(start + dt.timedelta(i)).strftime("%d%b%Y").upper() for i in range((end - start).days + 1)]
    if not all(x in have() for x in days):
        cfg["search_parameters"].update(startTime=str(start), endTime=str(end))
        json.dump(cfg, open(f"{d}/config.json", "w"), indent=2)
        subprocess.run(["python", "mdapi.py"], cwd=d, capture_output=True, text=True, input="y\n")
    h = have()
    print(start, "→", end, "| days with data:", sum(x in h for x in days), "/", len(days))
    start = end + dt.timedelta(days=1)
os.remove(f"{d}/config.json")      # don't leave the password on disk

## Extract AOD at each site
3×3-pixel box (~30 km) around each site; `-999` (cloud/missing) and values outside 0–5 are dropped; half-hourly values averaged to daily.

In [ ]:
sites = pd.read_csv(REPO + "data/soiling_sites.csv").dropna(subset=["lat", "lon"])
paths = sorted(glob.glob("/content/insat/**/*.h5", recursive=True))
with h5py.File(paths[0], "r") as f:
    lat, lon = f["latitude"][:], f["longitude"][:]
idx = {r.site: (np.abs(lat - r.lat).argmin(), np.abs(lon - r.lon).argmin()) for r in sites.itertuples()}

rows = []
for i, p in enumerate(paths):
    date = pd.to_datetime(os.path.basename(p)[6:15], format="%d%b%Y")
    with h5py.File(p, "r") as f:
        a = f["AOD"][0]
    for s, (yi, xi) in idx.items():
        w = a[max(yi - 1, 0):yi + 2, max(xi - 1, 0):xi + 2]
        w = w[(w > 0) & (w < 5)]
        if w.size:
            rows.append((date, s, w.mean()))
    if i % 500 == 0:
        print(i, "/", len(paths))

insat = pd.DataFrame(rows, columns=["date", "site", "aod_insat"]).groupby(["date", "site"]).mean().reset_index()
insat.to_csv("insat_site_daily_aod_2024.csv", index=False)

## ISRO vs NASA: which predicts measured soiling?
Fair comparison: NASA AOD is averaged over **exactly the same dry days** on which ISRO had a valid value.

In [ ]:
daily = pd.read_csv(REPO + "data/site_daily_2024.csv", parse_dates=["date"])
m = daily.merge(insat, on=["date", "site"], how="inner")
m["aod_merra2"] = m.aod_merra2.astype(float)
dry_m = m[m.rain_mm < 1]
aod_insat = dry_m.groupby("site").aod_insat.mean()
aod_merra_same = dry_m.groupby("site").aod_merra2.mean()

# dust deposition from notebook 02
dep_m = pd.read_csv(REPO + "data/merra2_site_monthly_dust_2024.csv", index_col="month")
dep_m.index = pd.PeriodIndex(dep_m.index, freq="M")
rain_m = daily.set_index("date").groupby("site").rain_mm.resample("ME").sum().unstack(0)
rain_m.index = rain_m.index.to_period("M")
dust_dry = dep_m.where(rain_m < 30).mean()

def evaluate2(proxy, name):
    d = sites.set_index("site").join(proxy.rename("x")).dropna(subset=["x", "rate_pct_per_day"])
    x, y = d.x.values, d.rate_pct_per_day.values
    b, a = np.polyfit(x, y, 1)
    loo = [np.polyval(np.polyfit(np.delete(x, i), np.delete(y, i), 1), x[i]) for i in range(len(x))]
    rmse_loo = np.sqrt(((np.array(loo) - y) ** 2).mean())
    base_loo = np.sqrt(np.mean([(y[i] - np.delete(y, i).mean()) ** 2 for i in range(len(y))]))
    print(f"{name}: rate = {a:.3f} + {b:.4f}·x | LOO RMSE = {rmse_loo:.3f} | LOO baseline = {base_loo:.3f}")

print(dry_m.groupby("site").size().rename("dry days with ISRO data"))
print(pd.DataFrame({"ISRO": aod_insat, "NASA": aod_merra_same}).round(3))
print("Daily ISRO–NASA AOD agreement r =", round(m[["aod_insat", "aod_merra2"]].corr().iloc[0, 1], 2))
evaluate2(aod_insat, "ISRO INSAT-3DR AOD")
evaluate2(aod_merra_same, "NASA MERRA-2 AOD (same days)")
evaluate2(dust_dry, "NASA MERRA-2 dust deposition")

## Check: is Chennai's high ISRO AOD caused by sea pixels?
Compare the original box, the single centre pixel, and a box ~22 km inland.

In [ ]:
yi, xi = np.abs(lat - 13.08).argmin(), np.abs(lon - 80.27).argmin()
xw = np.abs(lon - 80.07).argmin()
res = {"original 3×3 box": [], "single pixel": [], "inland 3×3 box": []}
for p in paths:
    date = pd.to_datetime(os.path.basename(p)[6:15], format="%d%b%Y")
    with h5py.File(p, "r") as f:
        a = f["AOD"][0]
    for name, w in [("original 3×3 box", a[yi-1:yi+2, xi-1:xi+2]),
                    ("single pixel",     a[yi:yi+1, xi:xi+1]),
                    ("inland 3×3 box",   a[yi-1:yi+2, xw-1:xw+2])]:
        w = w[(w > 0) & (w < 5)]
        if w.size:
            res[name].append((date, w.mean()))

dry_days = set(daily[(daily.site == "Chennai") & (daily.rain_mm < 1)].date)
for name, v in res.items():
    s = pd.DataFrame(v, columns=["date", "aod"]).groupby("date").aod.mean()
    s = s[s.index.isin(dry_days)]
    print(f"{name}: dry-day mean AOD = {s.mean():.3f} over {len(s)} days")